# Frame clustering and saliency

Compare Gaussian mixtures, K-means and agglomerative clustering on short event frames. Measure event capture and crop area, then inspect the selected region. Run from the repository root after following docs/DATASETS.md.


## 1. Environment

Install the project dependencies in the active notebook kernel.


In [ ]:
%pip install -r requirements-dev.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
import tonic

print("tonic version:", tonic.__version__)

## 2. Load a recording

Place the extracted gesture recordings in data/DVSGesture/ibmGestureTrain/ and ibmGestureTest/. See docs/DATASETS.md for the required layout. The following cell checks for local files before loading.


In [ ]:
from pathlib import Path

DATA_ROOT = Path("./data")
_extracted = DATA_ROOT / "DVSGesture" / "ibmGestureTrain"
_n_npy = len(list(_extracted.glob("**/*.npy"))) if _extracted.is_dir() else 0

if _n_npy < 100:
    raise SystemExit(
        f"DVSGesture recordings are missing: {_extracted} holds {_n_npy} .npy files, "
        "and at least 100 are needed.\nSee the notes above for the manual download."
    )

tonic.datasets.DVSGesture._is_file_present = lambda self: True

dataset = tonic.datasets.DVSGesture(save_to=str(DATA_ROOT), train=True)

events, label = dataset[0]
W, H = dataset.sensor_size[0], dataset.sensor_size[1]

x = events["x"].astype(np.int64)
y = events["y"].astype(np.int64)
t = events["t"].astype(np.int64)
p = events["p"].astype(np.int64)

print("recordings in split:", len(dataset))
print("event fields:", events.dtype.names)
print("number of events:", len(events))
print("sensor size (W, H):", W, H)
print("duration (ms):", (t.max() - t.min()) / 1000)
print("class label:", label)

## 3. Helpers

Cluster events by spatial coordinates. Rank clusters by event count, density or temporal-bin occupancy. Boxes use the 5th and 95th spatial percentiles to reduce outlier sensitivity.


In [ ]:
def cluster_frame(coords, method="kmeans", k=3, random_state=0):
    if len(coords) < k:
        return None
    if method == "kmeans":
        model = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    elif method == "agglomerative":
        model = AgglomerativeClustering(n_clusters=k)
    elif method == "gmm":
        model = GaussianMixture(n_components=k, random_state=random_state)
    else:
        raise ValueError(f"unknown method: {method}")
    return model.fit_predict(coords)


def fovea_box(points, margin=5):
    lo = np.percentile(points, margin, axis=0)
    hi = np.percentile(points, 100 - margin, axis=0)
    return int(lo[0]), int(lo[1]), int(hi[0]), int(hi[1])


def cluster_table(coords, times, labels, n_bins=5):
    edges = np.linspace(times.min(), times.max() + 1, n_bins + 1)
    rows = []
    for cid in np.unique(labels):
        m = labels == cid
        box = fovea_box(coords[m])
        area = max((box[2] - box[0]) * (box[3] - box[1]), 1.0)
        size = int(m.sum())
        occupied = np.unique(np.clip(np.digitize(times[m], edges) - 1, 0, n_bins - 1))
        rows.append(
            {
                "id": int(cid),
                "size": size,
                "density": size / area,
                "persistence": len(occupied) / n_bins,
                "box": box,
            }
        )
    return rows


def rank_clusters(coords, times, labels, by="size"):
    return sorted(cluster_table(coords, times, labels), key=lambda r: r[by], reverse=True)


def most_salient(coords, times, labels, by="size"):
    return rank_clusters(coords, times, labels, by=by)[0]

## 4. Inspect one frame

Fit three components to a 50 ms frame from the middle of the recording. The first panel shows cluster membership; the second marks the selected crop.


In [ ]:
WINDOW_US = 50_000
METHOD = "gmm"
K = 3

t0, tmax = t.min(), t.max()
start = t0 + (tmax - t0) // 2
mask = (t >= start) & (t < start + WINDOW_US)
xs, ys, ts = x[mask], y[mask], t[mask]
coords = np.column_stack([xs, ys]).astype(np.float32)

labels = cluster_frame(coords, method=METHOD, k=K)
fovea = most_salient(coords, ts, labels)
x0, y0, x1, y1 = fovea["box"]

frame = np.zeros((H, W), dtype=np.float32)
np.add.at(frame, (ys, xs), 1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
for cl in sorted(set(labels)):
    m = labels == cl
    ax1.scatter(xs[m], ys[m], s=3, label=f"cluster {cl}")
ax1.invert_yaxis()
ax1.set_aspect("equal")
ax1.set_title(f"{METHOD}, k={K}")
ax1.legend(markerscale=4, fontsize=8)

ax2.imshow(frame, cmap="hot")
ax2.add_patch(
    patches.Rectangle((x0, y0), x1 - x0, y1 - y0, lw=2, edgecolor="cyan", facecolor="none")
)
ax2.set_title(f"fovea = most salient cluster ({fovea['size']} events)")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 5. Process the recording

Apply the same clustering to successive frames. Inspect six snapshots spread across the recording.


In [ ]:
results = []
starts = np.arange(t0, tmax, WINDOW_US)

for s in starts:
    m = (t >= s) & (t < s + WINDOW_US)
    if int(m.sum()) < K:
        results.append((s, None, None, 0))
        continue
    xs_, ys_, ts_ = x[m], y[m], t[m]
    coords_ = np.column_stack([xs_, ys_]).astype(np.float32)
    labels_ = cluster_frame(coords_, method=METHOD, k=K)
    fovea_ = most_salient(coords_, ts_, labels_)
    fr = np.zeros((H, W), dtype=np.float32)
    np.add.at(fr, (ys_, xs_), 1.0)
    results.append((s, fr, fovea_["box"], fovea_["size"]))

filled = [r for r in results if r[1] is not None]
print(f"processed {len(results)} frames, {len(filled)} had enough events to cluster")

idx = np.linspace(0, len(filled) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, i in zip(axes.ravel(), idx):
    s, fr, box, nev = filled[i]
    bx0, by0, bx1, by1 = box
    ax.imshow(fr, cmap="hot")
    ax.add_patch(
        patches.Rectangle(
            (bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor="cyan", facecolor="none"
        )
    )
    ax.set_title(f"t = {(s - t0) / 1000:.0f} ms, fovea = {nev} ev", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Most salient cluster per frame  ({METHOD}, k={K})")
plt.tight_layout()
plt.show()

## 6. Frame length and cluster count

Sweep the frame window and component count. Compare median event capture, box-area fraction and capture per unit area. Smaller boxes are useful only if they retain relevant activity.


In [ ]:
def fovea_capture(xs_, ys_, ts_, k, method=METHOD):
    coords_ = np.column_stack([xs_, ys_]).astype(np.float32)
    labels_ = cluster_frame(coords_, method=method, k=k)
    if labels_ is None:
        return None
    box = most_salient(coords_, ts_, labels_)["box"]
    inside = (xs_ >= box[0]) & (xs_ <= box[2]) & (ys_ >= box[1]) & (ys_ <= box[3])
    box_frac = max((box[2] - box[0]) * (box[3] - box[1]), 1) / (W * H)
    return inside.mean(), box_frac


windows = [20_000, 35_000, 50_000, 75_000, 100_000]
ks = [2, 3, 4, 5]
rng = np.random.default_rng(0)

print(f"method: {METHOD}\n")
print(f"{'window ms':>10}{'k':>4}{'capture':>10}{'box frac':>10}{'concentration':>15}")
for window in windows:
    frame_starts = np.arange(t0, tmax, window)
    if len(frame_starts) > 25:
        frame_starts = rng.choice(frame_starts, 25, replace=False)
    for k in ks:
        caps, boxes = [], []
        for s in frame_starts:
            m = (t >= s) & (t < s + window)
            if int(m.sum()) < k:
                continue
            r = fovea_capture(x[m], y[m], t[m], k)
            if r is not None:
                caps.append(r[0])
                boxes.append(r[1])
        if caps:
            cap, bf = np.median(caps), np.median(boxes)
            print(f"{window / 1000:>10.0f}{k:>4}{cap:>10.2f}{bf:>10.3f}{cap / bf:>15.1f}")

## 7. Selection rules

Compare cluster size, density and temporal occupancy within a frame. Occupancy can saturate when all components are active across the window; notebook 03 evaluates persistence across frames.


In [ ]:
labels_mid = cluster_frame(coords, method=METHOD, k=K)
table = cluster_table(coords, ts, labels_mid)

print(f"{'cluster':>8}{'size':>8}{'density':>10}{'persistence':>13}")
for r in sorted(table, key=lambda r: r["id"]):
    print(f"{r['id']:>8}{r['size']:>8}{r['density']:>10.3f}{r['persistence']:>13.2f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, by in zip(axes, ["size", "density", "persistence"]):
    pick = most_salient(coords, ts, labels_mid, by=by)
    bx0, by0, bx1, by1 = pick["box"]
    ax.imshow(frame, cmap="hot")
    ax.add_patch(
        patches.Rectangle(
            (bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor="cyan", facecolor="none"
        )
    )
    ax.set_title(f"salient by {by}: cluster {pick['id']}")
    ax.axis("off")
plt.tight_layout()
plt.show()


def fovea_centres(by):
    centres = []
    for s in np.arange(t0, tmax, WINDOW_US):
        m = (t >= s) & (t < s + WINDOW_US)
        if int(m.sum()) < K:
            continue
        coords_ = np.column_stack([x[m], y[m]]).astype(np.float32)
        box = most_salient(coords_, t[m], cluster_frame(coords_, method=METHOD, k=K), by=by)["box"]
        centres.append(((box[0] + box[2]) / 2, (box[1] + box[3]) / 2))
    return np.array(centres)


for by in ["size", "density", "persistence"]:
    c = fovea_centres(by)
    jump = np.hypot(*(c[1:] - c[:-1]).T).mean()
    print(f"{by:>12}: mean fovea jump = {jump:.1f} px/frame")

## 8. One region or several

Compare the top-ranked cluster with the top three candidates. The crop pipeline uses one region; multiple candidates are used by the tracking experiment.


In [ ]:
top = rank_clusters(coords, ts, labels_mid, by="size")[:3]
colours = ["cyan", "yellow", "magenta"]

fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 5))
for ax, keep, title in [(axa, 1, "single fovea"), (axb, 3, "top 3 clusters")]:
    ax.imshow(frame, cmap="hot")
    for r, colour in zip(top[:keep], colours):
        bx0, by0, bx1, by1 = r["box"]
        ax.add_patch(
            patches.Rectangle(
                (bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor=colour, facecolor="none"
            )
        )
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Clustering comparison

Compare event capture, box area and runtime for each method at the same component count. Event count alone does not measure spatial compactness. Historical measurements and revisions are retained in FOVEANET_LOG.txt.


In [ ]:
import time

sweep_starts = np.arange(t0, tmax, WINDOW_US)

print(
    f"{'method':>15}{'k':>3}{'capture':>10}{'box frac':>10}{'concentration':>15}{'sec/frame':>12}"
)
for method in ["kmeans", "gmm", "agglomerative"]:
    for k in [2, 3, 4]:
        caps, boxes, elapsed, n = [], [], 0.0, 0
        for s in sweep_starts:
            m = (t >= s) & (t < s + WINDOW_US)
            if int(m.sum()) < 50:
                continue
            xs_, ys_, ts_ = x[m], y[m], t[m]
            coords_ = np.column_stack([xs_, ys_]).astype(np.float32)
            if len(coords_) > 4000:
                sel = np.random.default_rng(0).choice(len(coords_), 4000, replace=False)
                coords_, xs_, ys_, ts_ = coords_[sel], xs_[sel], ys_[sel], ts_[sel]
            t_start = time.perf_counter()
            lbl = cluster_frame(coords_, method=method, k=k)
            elapsed += time.perf_counter() - t_start
            n += 1
            box = most_salient(coords_, ts_, lbl)["box"]
            inside = (xs_ >= box[0]) & (xs_ <= box[2]) & (ys_ >= box[1]) & (ys_ <= box[3])
            caps.append(inside.mean())
            boxes.append(max((box[2] - box[0]) * (box[3] - box[1]), 1) / (W * H))
        cap, bf = np.median(caps), np.median(boxes)
        print(f"{method:>15}{k:>3}{cap:>10.2f}{bf:>10.3f}{cap / bf:>15.1f}{elapsed / n:>12.4f}")

## 10. Related experiments

Notebook 02 links detections across frames. Notebook 03 carries Gaussian-mixture parameters forward and accumulates per-component persistence.
